## EXP 5: Char-level text generation (SimpleRNN)

In [1]:
import numpy as np, tensorflow as tf
tf.keras.utils.set_random_seed(42)

text = open(r"C:\Users\nande\OneDrive\Desktop\Edu.in\24AIPL401-DeepLearning\DataSet\Exp5\shakespeare.txt", encoding="utf-8").read()      # <- set your path
vocab = sorted(set(text)); V = len(vocab)                      # FIX: V computed, not hard-coded 65
c2i = {c: i for i, c in enumerate(vocab)}
ids = np.array([c2i[c] for c in text], dtype="int32")
print("Corpus length:", len(text), "| Vocab size:", V)

# every overlapping window of 101 chars: first 100 = input, last 100 = target (shifted by 1)
w = np.lib.stride_tricks.sliding_window_view(ids, 101)
ds = tf.data.Dataset.from_tensor_slices((w[:, :-1], w[:, 1:])).shuffle(10000).batch(64, drop_remainder=True)

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(V, 128),
    tf.keras.layers.SimpleRNN(256, return_sequences=True),
    tf.keras.layers.Dense(V)])
model.compile(optimizer="adam", loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
model.summary()
model.fit(ds, epochs=1)

def generate(start, n=200):
    out = start
    for _ in range(n):
        x = np.array([[c2i[c] for c in out[-100:]]])
        out += vocab[model.predict(x, verbose=0)[0, -1].argmax()]   # greedy: most likely next char
    return out

print("\nGenerated Text:\n", generate("Shall ", 200))

Corpus length: 98328 | Vocab size: 73
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, None, 128)         9344      
                                                                 
 simple_rnn (SimpleRNN)      (None, None, 256)         98560     
                                                                 
 dense (Dense)               (None, None, 73)          18761     
                                                                 
Total params: 126,665
Trainable params: 126,665
Non-trainable params: 0
_________________________________________________________________
1534/1534 [==============================] - 91s 59ms/step - loss: 1.5305

Generated Text:
 Shall of thise thoughts and thee mourn,
But thou art the fairest on thy sweet self see hath to self soul the soul in thy see the world my self and what to mine of men sick when I love 

## EXP 6: IMDB sentiment (LSTM)

In [2]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
tf.keras.utils.set_random_seed(42)

(xtr, ytr), (xte, yte) = imdb.load_data(num_words=10000)
xtr, xte = pad_sequences(xtr, maxlen=200), pad_sequences(xte, maxlen=200)

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(10000, 128),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(1, activation="sigmoid")])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(xtr, ytr, epochs=3, batch_size=64)
print("Test Accuracy:", model.evaluate(xte, yte)[1])

# custom sentence. Keras IMDB ids are word_index + 3 (0=pad, 1=start, 2=unknown)
wi = imdb.get_word_index()
def encode(sentence):
    ids = [1]                                                   # FIX: start token like the training data
    for w in sentence.lower().split():
        i = wi.get(w.strip(".,!?"), None)
        ids.append(i + 3 if i is not None and i + 3 < 10000 else 2)   # FIX: unknown -> 2, ids >= 10000 -> 2
    return pad_sequences([ids], maxlen=200)

sentence = "This movie was super"
p = model.predict(encode(sentence))[0][0]
print("Sentence:", sentence, "\nSentiment:", "Positive" if p >= 0.5 else "Negative", "\nScore:", p)

17464789/17464789 [==============================] - 3s 0us/step
Epoch 1/3
391/391 [==============================] - 37s 93ms/step - loss: 0.4034 - accuracy: 0.8111
Epoch 2/3
391/391 [==============================] - 47s 120ms/step - loss: 0.2269 - accuracy: 0.9120
Epoch 3/3
782/782 [==============================] - 22s 27ms/step - loss: 0.3597 - accuracy: 0.8553
Test Accuracy: 0.8552799820899963
1/1 [==============================] - 0s 329ms/step
Sentence: This movie was super 
Sentiment: Positive 
Score: 0.60684925


## EXP 7: POS tagging (Encoder-Decoder LSTM)

In [ ]:
import nltk, numpy as np
from nltk.corpus import treebank
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
nltk.download("treebank")

sents = treebank.tagged_sents()
X = [[w for w, t in s] for s in sents]
Y = [[t for w, t in s] for s in sents]
Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.2, random_state=42)

wtok = Tokenizer(lower=True, oov_token="<UNK>"); wtok.fit_on_texts(Xtr)     
ttok = Tokenizer(lower=False, filters="");       ttok.fit_on_texts(Ytr)
wv, tv = len(wtok.word_index) + 1, len(ttok.word_index) + 1
max_len = max(len(s) for s in Xtr)

def pad(tok, data):
    return pad_sequences(tok.texts_to_sequences(data), maxlen=max_len, padding="post", truncating="post")
Xtr_p, Xte_p, Ytr_p, Yte_p = pad(wtok, Xtr), pad(wtok, Xte), pad(ttok, Ytr), pad(ttok, Yte)
print("Vocab:", wv, "| Tags:", tv, "| Max len:", max_len, "| Input:", Xtr_p.shape)

inp = Input((max_len,))
enc_out, h, c = LSTM(128, return_sequences=True, return_state=True)(Embedding(wv, 128, mask_zero=True)(inp))
dec_out = LSTM(128, return_sequences=True)(enc_out, initial_state=[h, c])
model = Model(inp, Dense(tv, activation="softmax")(dec_out))
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
model.fit(Xtr_p, Ytr_p[..., None], validation_split=0.1, epochs=5, batch_size=32)
loss, acc = model.evaluate(Xte_p, Yte_p[..., None])
print("Test Loss:", loss, "| Test Accuracy:", acc)

id2tag = {v: k for k, v in ttok.word_index.items()}
pred = model.predict(Xte_p).argmax(-1)
i = 782                                                         
print("\nWords:", Xte[i], "\nActual:", Yte[i], "\nPredicted:", [id2tag.get(t, "UNK") for t in pred[i][:len(Xte[i])]])

[nltk_data] Downloading package treebank to
[nltk_data]     C:\Users\nande\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\treebank.zip.


Vocab: 10140 | Tags: 47 | Max len: 271 | Input: (3131, 271)
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 271)]        0           []                               
                                                                                                  
 embedding_2 (Embedding)        (None, 271, 128)     1297920     ['input_1[0][0]']                
                                                                                                  
 lstm_1 (LSTM)                  [(None, 271, 128),   131584      ['embedding_2[0][0]']            
                                 (None, 128),                                                     
                                 (None, 128)]                                                     
                                  

## EXP 8: English -> Tamil translation (Seq2Seq LSTM)

In [ ]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

pairs = []
for line in open(r"C:\Users\nande\OneDrive\Desktop\Edu.in\24AIPL401-DeepLearning\DataSet\Exp8\eng_tamil_pairs.txt", encoding="utf-8"):     # <- set your path
    p = line.strip().split("\t", 1)
    if len(p) == 2: pairs.append((p[0].strip(), p[1].strip()))
print("Pairs:", len(pairs))
pairs = list(dict.fromkeys(pairs))                              
print("Unique pairs:", len(pairs))
english = [e for e, t in pairs]
tamil = [f"<start> {t} <end>" for e, t in pairs]

def tokenize(texts):
    tok = Tokenizer(filters=""); tok.fit_on_texts(texts)
    seq = tok.texts_to_sequences(texts); L = max(len(s) for s in seq)
    return tok, pad_sequences(seq, maxlen=L, padding="post"), L
et, eng, max_eng = tokenize(english)
tt, tam, max_tam = tokenize(tamil)
ev, tv = len(et.word_index) + 1, len(tt.word_index) + 1
print("Max len:", max_eng, max_tam, "| Vocab:", ev, tv)

dec_in_data, dec_out_data = tam[:, :-1], tam[:, 1:]             

# Encoder
enc_in = Input((max_eng,))
_, h, c = LSTM(128, return_state=True)(Embedding(ev, 64, mask_zero=True)(enc_in))   
# Decoder
dec_in = Input((max_tam - 1,))
dec = LSTM(128, return_sequences=True)(Embedding(tv, 64, mask_zero=True)(dec_in), initial_state=[h, c])  # FIX
model = Model([enc_in, dec_in], Dense(tv, activation="softmax")(dec))
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
model.fit([eng, dec_in_data], dec_out_data[..., None], epochs=50, batch_size=32, verbose=1)

rev = {i: w for w, i in tt.word_index.items()}
start, end = tt.word_index["<start>"], tt.word_index["<end>"]
def translate(sentence):
    src = pad_sequences(et.texts_to_sequences([sentence]), maxlen=max_eng, padding="post")
    tgt = [start]
    for _ in range(max_tam - 1):                                # generate one word at a time (greedy)
        t = pad_sequences([tgt], maxlen=max_tam - 1, padding="post")
        nxt = model.predict([src, t], verbose=0)[0, len(tgt) - 1].argmax()
        if nxt in (0, end): break
        tgt.append(nxt)
    return " ".join(rev[i] for i in tgt[1:])

for s in ["Get down.", "Goodbye and take care.", "Let's meet again soon.", "Good morning.", "Thank you.", "How are you?", "hospital"]:
    print("\nEnglish :", s, "\nTamil   :", translate(s))

Pairs: 4611
Unique pairs: 4198
Max len: 16 14 | Vocab: 3271 4563
Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_4 (InputLayer)           [(None, 16)]         0           []                               
                                                                                                  
 input_5 (InputLayer)           [(None, 13)]         0           []                               
                                                                                                  
 embedding_5 (Embedding)        (None, 16, 64)       209344      ['input_4[0][0]']                
                                                                                                  
 embedding_6 (Embedding)        (None, 13, 64)       292032      ['input_5[0][0]']                
                           

## EXP 9: GAN data augmentation (Fashion-MNIST)

In [ ]:
import numpy as np, tensorflow as tf, matplotlib.pyplot as plt
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(42)

(xr, ytr), (xt, yte) = tf.keras.datasets.fashion_mnist.load_data()
xr, xt = xr[..., None].astype("float32"), xt[..., None].astype("float32")
x_gan = (xr - 127.5) / 127.5                                    
x_cls, x_test = xr / 255.0, xt / 255.0                          

G = tf.keras.Sequential([layers.Input((100,)),
    layers.Dense(7 * 7 * 256, use_bias=False), layers.BatchNormalization(), layers.LeakyReLU(), layers.Reshape((7, 7, 256)),
    layers.Conv2DTranspose(128, 4, 2, padding="same", use_bias=False), layers.BatchNormalization(), layers.LeakyReLU(),
    layers.Conv2DTranspose(64, 4, 2, padding="same", use_bias=False), layers.BatchNormalization(), layers.LeakyReLU(),
    layers.Conv2D(1, 3, padding="same", activation="tanh")])

D = tf.keras.Sequential([layers.Input((28, 28, 1)),            
    layers.Conv2D(64, 4, 2, padding="same"), layers.LeakyReLU(0.2), layers.Dropout(0.3),
    layers.Conv2D(128, 4, 2, padding="same"), layers.LeakyReLU(0.2), layers.Dropout(0.3),
    layers.Flatten(), layers.Dense(1, activation="sigmoid")])
D.compile(optimizer=tf.keras.optimizers.Adam(0.0002, 0.5), loss="binary_crossentropy")

D.trainable = False                                             
GAN = tf.keras.Sequential([G, D])
GAN.compile(optimizer=tf.keras.optimizers.Adam(0.0002, 0.5), loss="binary_crossentropy")

noise = lambda n: np.random.normal(0, 1, (n, 100)).astype("float32")
for epoch in range(50):
    for i in range(0, len(x_gan), 128):
        real = x_gan[i:i + 128]; n = len(real)
        fake = G(noise(n), training=True).numpy()               
        D.trainable = True
        D.train_on_batch(real, np.full((n, 1), 0.9))           
        D.train_on_batch(fake, np.zeros((n, 1)))
        D.trainable = False
        GAN.train_on_batch(noise(n), np.ones((n, 1)))           
    print("Epoch:", epoch + 1)

gen = (G.predict(noise(16), verbose=0) + 1) / 2                 
plt.figure(figsize=(8, 8))
for i in range(16):
    plt.subplot(4, 4, i + 1); plt.imshow(gen[i].squeeze(), cmap="gray"); plt.axis("off")
plt.suptitle("GAN Generated Images"); plt.tight_layout(); plt.show()

num_fake = 5000
fake_images = (G.predict(noise(num_fake), verbose=0) + 1) / 2
print("Generated images:", fake_images.shape)

def classifier():
    m = tf.keras.Sequential([layers.Input((28, 28, 1)),
        layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Flatten(), layers.Dense(64, activation="relu"), layers.Dense(10, activation="softmax")])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

print("Training original classifier...")
m_orig = classifier(); m_orig.fit(x_cls, ytr, epochs=3, batch_size=128)
acc_orig = m_orig.evaluate(x_test, yte, verbose=0)[1]


fake_labels = m_orig.predict(fake_images, verbose=0).argmax(1)
# fake_labels = np.random.choice(ytr, num_fake)

x_aug = np.concatenate([x_cls, fake_images]); y_aug = np.concatenate([ytr, fake_labels])
idx = np.random.permutation(len(x_aug)); x_aug, y_aug = x_aug[idx], y_aug[idx]
print("Augmented data:", x_aug.shape)

print("Training augmented classifier...")
m_aug = classifier(); m_aug.fit(x_aug, y_aug, epochs=3, batch_size=128)
acc_aug = m_aug.evaluate(x_test, yte, verbose=0)[1]

print(f"\nOriginal Accuracy: {acc_orig*100:.2f} %\nGAN Augmented Accuracy: {acc_aug*100:.2f} %\nChange: {(acc_aug-acc_orig)*100:.2f} %")
plt.bar(["Original", "GAN Augmented"], [acc_orig * 100, acc_aug * 100])
plt.ylabel("Accuracy (%)"); plt.title("Classification Accuracy"); plt.ylim(0, 100); plt.show()